# 02 ? Candidate-table contract and validation

**Owner:** Person B ? Modelling Lead

**Purpose:** Validate Person A's generated candidate tables and create a normalized interface for modelling.

## Input schema from Person A

- `trip_id`: trip identifier
- `point_idx`: GPS observation index
- `t`: timestamp
- `arc_id`: candidate road arc ID
- `perp_dist`: perpendicular distance in projected metres
- `proj_frac`: projection fraction along candidate arc
- `label`: 1 = positive, 0 = negative, -1 = unlabelled

The label alignment is performed upstream by Person A using the reference route and GPS observations. The candidate-generation run uses a 50 m candidate radius and 50 m label-distance threshold.

Unlabelled rows must not be used for supervised fitting or labelled evaluation. They are retained for later inference and export.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
RAW_TRAIN = REPO_ROOT / "data/processed/candidates_train.csv"
RAW_STUDENT = REPO_ROOT / "data/processed/candidates_student_test.csv"

TRAIN_CSV = REPO_ROOT / "data/processed/train_candidates.csv"
STUDENT_CSV = REPO_ROOT / "data/processed/student_test_candidates.csv"

RENAME = {
    "point_idx": "observation_idx",
    "arc_id": "candidate_arc_id",
    "label": "is_match",
}

REQUIRED_RAW = {
    "trip_id", "point_idx", "t", "arc_id",
    "perp_dist", "proj_frac", "label"
}
TARGET = "is_match"
FEATURES = ["perp_dist", "proj_frac"]


In [ ]:
def load_and_normalize(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing candidate input: {path}")

    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"{path.name} is empty")

    missing = REQUIRED_RAW - set(df.columns)
    if missing:
        raise ValueError(f"Missing input columns: {sorted(missing)}")

    if df[list(REQUIRED_RAW)].isna().any().any():
        raise ValueError("Candidate table contains missing required values")

    df = df.rename(columns=RENAME)

    for col in ["observation_idx", "candidate_arc_id", "is_match"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")

    labels = set(df[TARGET].unique())
    if not labels.issubset({-1, 0, 1}):
        raise ValueError(f"Unexpected labels: {labels}")

    if not np.isfinite(df[FEATURES].to_numpy(dtype=float)).all():
        raise ValueError("Feature columns contain non-finite values")

    if ((df["proj_frac"] < -1e-6) | (df["proj_frac"] > 1 + 1e-6)).any():
        raise ValueError("proj_frac contains values outside [0, 1]")

    return df

train_all = load_and_normalize(RAW_TRAIN)
student_all = load_and_normalize(RAW_STUDENT)

print("Training:", train_all.shape, "| trips:", train_all.trip_id.nunique())
print("Student-test:", student_all.shape, "| trips:", student_all.trip_id.nunique())
print("Training labels:")
display(train_all[TARGET].value_counts().sort_index())
print("Student-test labels:")
display(student_all[TARGET].value_counts().sort_index())


In [ ]:
def validate_partition(df, name):
    key = ["trip_id", "observation_idx", "candidate_arc_id"]
    duplicate_count = int(df.duplicated(key).sum())
    if duplicate_count:
        raise ValueError(f"{name}: {duplicate_count} duplicate candidate keys")

    labelled = df[df[TARGET].isin([0, 1])].copy()
    unlabelled = df[df[TARGET] == -1].copy()

    if labelled.empty or labelled[TARGET].nunique() != 2:
        raise ValueError(f"{name}: supervised data must contain both classes")

    obs = df.groupby(["trip_id", "observation_idx"])[TARGET]
    obs_summary = obs.agg(
        candidate_count="size",
        max_label="max",
        min_label="min"
    )

    labelled_obs = obs_summary[obs_summary["max_label"] != -1]
    positive_obs = int((labelled_obs["max_label"] == 1).sum())
    no_positive_obs = int((labelled_obs["max_label"] == 0).sum())

    print(f"\\n===== {name} =====")
    print("Rows:", len(df))
    print("Trips:", df.trip_id.nunique())
    print("Labelled candidate rows:", len(labelled))
    print("Unlabelled candidate rows:", len(unlabelled))
    print("Unlabelled fraction:", round(len(unlabelled) / len(df), 5))
    print("Duplicate keys:", duplicate_count)
    print("Labelled observations with positive candidate:", positive_obs)
    print("Labelled observations without positive candidate:", no_positive_obs)
    display(obs_summary["candidate_count"].describe())

    return labelled, unlabelled

train_labelled, train_unlabelled = validate_partition(train_all, "TRAIN")
student_labelled, student_unlabelled = validate_partition(student_all, "STUDENT TEST")

print("\\nFeatures:", FEATURES)
print("Target:", TARGET)
print("Group:", "trip_id")


In [ ]:
# Save normalized tables with the schema expected by downstream notebooks.
# All rows are retained, including is_match = -1.
TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)

train_all.to_csv(TRAIN_CSV, index=False)
student_all.to_csv(STUDENT_CSV, index=False)

manifest = {
    "feature_columns": FEATURES,
    "target_column": TARGET,
    "group_column": "trip_id",
    "candidate_key": ["trip_id", "observation_idx", "candidate_arc_id"],
    "input_columns": list(train_all.columns),
    "label_mapping": {"1": "positive", "0": "negative", "-1": "unlabelled"},
    "unlabelled_policy": "Exclude -1 from supervised fitting and labelled metrics; retain for inference.",
    "label_alignment": "Upstream Person A route-to-GPS alignment; candidate radius 50m; label distance threshold 50m.",
    "feature_policy": "Initial baseline uses perp_dist and proj_frac only. IDs and raw timestamp are excluded."
}

manifest_path = REPO_ROOT / "configs/person_b_feature_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", TRAIN_CSV)
print("Saved:", STUDENT_CSV)
print("Saved:", manifest_path)
print("Normalized schema validation complete.")


## Stop/go checklist

- [x] Actual Person A candidate files loaded.
- [x] Source columns mapped to the Person B interface.
- [x] Unlabelled `-1` rows identified and preserved.
- [x] Duplicate candidate keys checked.
- [x] Numeric baseline features validated.
- [ ] Group-aware model comparison completed.
- [ ] Bayesian optimisation completed.
- [ ] Student-test evaluation completed.

**Important:** This notebook validates the candidate-table interface. It does not establish model performance or independently prove the correctness of upstream route labels.
